# 🔥 Bushfire History — Australia (2016–2021)

**Source:** [ABARES — Forests Australia](https://www.agriculture.gov.au/abares/forestsaustralia/forest-data)

Broad-scale fire history across Australian forests. Each row is a forest polygon 
with tenure info, state, and annual burn records (planned/prescribed vs unplanned/wildfire) 
for 5 financial years: 2016–17 through 2020–21.

**ML Task:** Multiclass classification — predict total burn count (0–5) from tenure, category, state, and per-year burn patterns.

**Dataset path:** 


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import pickle
from datetime import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

print("✅ Libraries loaded")

In [ ]:
BASE = Path.cwd()
RAW = BASE / "raw"
PROCESSED = BASE / "processed"
FEATURES = BASE / "features"

df = pd.read_csv(RAW / "Fire_For16-21_Attributes.csv", encoding="utf-8")
print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)

In [ ]:
print("--- EDA ---")
print(f"Shape: {df.shape}")
print(f"
Dtypes:
{df.dtypes}")
print(f"
Missing values:
{df.isnull().sum()}")
print(f"
Duplicate rows: {df.duplicated().sum()}")
print(f"
Numeric stats:
{df.describe()}")
print(f"
STATE distribution:
{df["STATE"].value_counts()}")
print(f"
FOR_BURNS distribution:
{df["FOR_BURNS"].value_counts().sort_index()}")

In [ ]:
print("🧹 CLEANING")
print("=" * 60)

df_clean = df.copy()

# 1. Drop ID columns
df_clean = df_clean.drop(columns=["OID", "VALUE"], errors="ignore")
print("✓ Dropped OID, VALUE (index columns)")

# 2. Standardize column names
df_clean.columns = (df_clean.columns.str.strip()
                    .str.lower()
                    .str.replace(r"[^a-z0-9_]", "_", regex=True)
                    .str.replace(r"_+", "_", regex=True)
                    .str.strip("_"))
print("✓ Standardized column names")

# 3. Replace blanks with NaN
for col in df_clean.select_dtypes(include=["object"]).columns:
    df_clean[col] = df_clean[col].astype(str).str.strip().replace("", None)

# 4. Remove duplicates
dupes = df_clean.duplicated().sum()
df_clean = df_clean.drop_duplicates()
print(f"✓ Removed {dupes:,} duplicate rows")

# 5. Fill missing values
for col in df_clean.select_dtypes(include=[np.number]).columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())

for col in df_clean.select_dtypes(include=["object"]).columns:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna("Unknown")

# 6. Fix FOR_BURNS -9
if "for_burns" in df_clean.columns:
    n_neg9 = (df_clean["for_burns"] == -9).sum()
    df_clean["for_burns"] = df_clean["for_burns"].clip(lower=0)
    print(f"✓ Capped {n_neg9} FOR_BURNS values of -9 → 0")

print(f"
✅ Cleaned: {df_clean.shape[0]:,} rows, {df_clean.shape[1]} cols, {df_clean.isnull().sum().sum()} missing")

In [ ]:
print("🔧 FEATURE ENGINEERING")
print("=" * 60)

df_feat = df_clean.copy()

fire_year_cols = ["fire_1617", "fire_1718", "fire_1819", "fire_1920", "fire_2021"]

for col in fire_year_cols:
    df_feat[f"{col}_any_fire"] = df_feat[col].notna().astype(int)
    df_feat[f"{col}_unplanned"] = (df_feat[col] == "U").astype(int)
    df_feat[f"{col}_planned"] = (df_feat[col] == "P").astype(int)

fire_bool = [c for c in df_feat.columns if c.endswith("_any_fire")]
unplanned = [c for c in df_feat.columns if c.endswith("_unplanned")]
planned = [c for c in df_feat.columns if c.endswith("_planned")]

df_feat["total_years_burned"] = df_feat[fire_bool].sum(axis=1)
df_feat["total_unplanned_burns"] = df_feat[unplanned].sum(axis=1)
df_feat["total_planned_burns"] = df_feat[planned].sum(axis=1)
df_feat["any_unplanned_fire"] = (df_feat["total_unplanned_burns"] > 0).astype(int)
df_feat["always_burned"] = (df_feat["total_years_burned"] == 5).astype(int)

# Drop redundant columns
df_feat = df_feat.drop(columns=["all_fire", "for_burn_t"], errors="ignore")
df_feat = df_feat.drop(columns=fire_year_cols)

# One-hot encode categoricals
cat_cols = df_feat.select_dtypes(include=["object"]).columns.tolist()
df_feat = pd.get_dummies(df_feat, columns=cat_cols, drop_first=True)
for col in df_feat.select_dtypes(include=["bool"]).columns:
    df_feat[col] = df_feat[col].astype(int)

print(f"✓ Created per-year fire indicators (any, unplanned, planned)")
print(f"✓ Created aggregate: total_years_burned, total_unplanned_burns, total_planned_burns")
print(f"✓ Created binary flags: any_unplanned_fire, always_burned")
print(f"✓ One-hot encoded {len(cat_cols)} categorical columns")
print(f"
✅ Shape: {df_feat.shape[0]:,} rows × {df_feat.shape[1]} features")

In [ ]:
PROCESSED.mkdir(exist_ok=True)
clean_path = PROCESSED / "bushfire-history_clean.csv"
df_feat.to_csv(clean_path, index=False)
print(f"✅ Saved cleaned data → {clean_path}")

In [ ]:
print("🔧 ML PREP")
print("=" * 60)

# Convert any remaining bool
for col in df_feat.columns:
    if df_feat[col].dtype == bool:
        df_feat[col] = df_feat[col].astype(int)

target = "for_burns"
X = df_feat.select_dtypes(include=[np.number]).copy()
feat_cols = [c for c in X.columns if c != target]
y = df_feat[target]
X = X[feat_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

FEATURES.mkdir(exist_ok=True)
pd.DataFrame(X_train_scaled, columns=feat_cols).to_csv(FEATURES / "X_train_scaled.csv", index=False)
pd.DataFrame(X_test_scaled, columns=feat_cols).to_csv(FEATURES / "X_test_scaled.csv", index=False)
pd.DataFrame(y_train).to_csv(FEATURES / "y_train.csv", index=False, header=[target])
pd.DataFrame(y_test).to_csv(FEATURES / "y_test.csv", index=False, header=[target])

with open(FEATURES / "scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open(FEATURES / "feature_names.json", "w") as f:
    json.dump(feat_cols, f, indent=2)

print(f"✓ Train: {len(X_train):,} × {len(feat_cols)}")
print(f"✓ Test:  {len(X_test):,} × {len(feat_cols)}")
print(f"✓ Target: {target} ({y.nunique()} classes: {sorted(y.unique())})")
print(f"✅ ML files saved to features/")

In [ ]:
# Verify saved files load back correctly
verify = pd.read_csv(FEATURES / "X_train_scaled.csv")
print(f"X_train loaded: {verify.shape}")
verify_y = pd.read_csv(FEATURES / "y_train.csv")
print(f"y_train loaded: {verify_y.shape}")
print(f"Target distribution:
{verify_y[target].value_counts().sort_index()}")
print(f"
✅ Verification passed — all files load correctly")